# Imports

In [ ]:
import os
os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")

import json
import math

import numpy as np
import pandas as pd
import torch

from uavloc.features.segmentation import build_segmenter, load_segmenter
from uavloc.segmentation.eval import VAL_SETS, load_val_sets
from uavloc.segmentation.losses import CURRENT_LOSS, StructureLossConfig, ce_dice_loss, structure_loss_terms
import uavloc.segmentation.village_seg as vs
from uavloc.track import REPO_ROOT, resolve_device, set_seed

# Params

In [ ]:
DEVICE = "auto"
SEED = 0
CHECK_SEED = 5
LOSS_TOL = 1e-6
MIOU_TOL = 1e-3
SAT_CROP_PX, SAT_BATCH = 512, 4
DRONE_CROP_PX, DRONE_BATCH = 256, 8
DRONE_GSD_M = 0.10
RARE_CROP_P = 0.3
GAP_WEIGHT = 3.0
INSTANCE_WEIGHT = 1.0
TVERSKY_WEIGHT = 1.0
SKELETON_WEIGHT = 1.0
TRACK_TVERSKY_WEIGHT = 1.0

In [ ]:
SPEED_POC = REPO_ROOT / "pocs" / "mobilenet-lraspp-speed"
DOMAINS = ("satellite", "drone")

# Data

In [ ]:
set_seed(SEED)
device = resolve_device(DEVICE)
data = vs.load_village_data()
samplers = {"satellite": vs.SatelliteSampler(data.sat_train, SAT_CROP_PX, RARE_CROP_P),
            "drone": vs.DroneSampler(data.drone_train, DRONE_GSD_M, DRONE_CROP_PX, RARE_CROP_P)}
batch_sizes = {"satellite": SAT_BATCH, "drone": DRONE_BATCH}
device

In [ ]:
batches = {d: vs.batch_of(samplers[d], np.random.default_rng(CHECK_SEED), batch_sizes[d]) for d in DOMAINS}
{d: (tuple(x.shape), sorted(y.unique().tolist())) for d, (x, y) in batches.items()}

# Model

In [ ]:
reference = {d: load_segmenter(SPEED_POC / f"model-lraspp-bn-{d}.pt", device, architecture="lraspp") for d in DOMAINS}

In [ ]:
fresh = build_segmenter("lraspp-mobilenetv3", pretrained=True).to(device)

In [ ]:
building_config = StructureLossConfig(gap_weight=GAP_WEIGHT, instance_weight=INSTANCE_WEIGHT,
                                      tversky_weight=TVERSKY_WEIGHT)
structure_config = StructureLossConfig(gap_weight=GAP_WEIGHT, instance_weight=INSTANCE_WEIGHT,
                                       tversky_weight=TVERSKY_WEIGHT, skeleton_weight=SKELETON_WEIGHT,
                                       track_tversky_weight=TRACK_TVERSKY_WEIGHT)

# Run

In [ ]:
def logits_of(net: torch.nn.Module, domain: str) -> tuple[torch.Tensor, torch.Tensor]:
    """The domain's check batch through `net`, and its labels, on the device."""
    x, y = batches[domain]
    with torch.no_grad():
        return net.eval()(x.to(device)), y.to(device)

In [ ]:
def current_vs_ce_dice(domain: str) -> float:
    """|structure_loss_terms(current)["loss"] - ce_dice_loss| on a real batch of `domain`."""
    logits, y = logits_of(reference[domain], domain)
    return abs(structure_loss_terms(logits, y, CURRENT_LOSS)["loss"].item() - ce_dice_loss(logits, y).item())

In [ ]:
check_a = {d: current_vs_ce_dice(d) for d in DOMAINS}
assert all(diff <= LOSS_TOL for diff in check_a.values()), check_a
check_a

In [ ]:
vs.check_round3_splits(data)
{"sat_train": len(data.sat_train), "drone_train": len(data.drone_train), "drone_val": len(data.drone_val)}

In [ ]:
broken = vs.VillageData(data.sat, data.sat_train[1:], data.drone_train, data.drone_val)
try:
    vs.check_round3_splits(broken)
    raise AssertionError("a dropped satellite image went unnoticed")
except ValueError as error:
    check_b_negative = str(error)
check_b_negative

In [ ]:
reference_scores = json.loads((SPEED_POC / "results-lraspp.json").read_text())["final_scores"]
val_sets = load_val_sets(vs.SplitConfig().dataset)

In [ ]:
module_scores = vs.val_metrics(reference["satellite"], val_sets, device)
module_scores.update(vs.drone_val_metrics(reference["drone"], data.drone_val, DRONE_GSD_M, device))

In [ ]:
check_c = pd.DataFrame({name: {"results-lraspp.json": reference_scores[name]["miou"],
                               "module": module_scores[f"miou_{name}"]} for name in [*VAL_SETS, vs.DRONE_VAL_NAME]}).T
check_c["abs_diff"] = (check_c["module"] - check_c["results-lraspp.json"]).abs()
assert (check_c["abs_diff"] <= MIOU_TOL).all(), check_c
check_c.round(4)

In [ ]:
def finite_terms(net: torch.nn.Module, domain: str, config: StructureLossConfig) -> dict[str, float]:
    """Every loss term of `config` on the domain's batch; backward must leave every gradient finite too."""
    x, y = batches[domain]
    net.train().zero_grad()
    terms = structure_loss_terms(net(x.to(device)), y.to(device), config)
    terms["loss"].backward()
    grads_finite = all(torch.isfinite(p.grad).all().item() for p in net.parameters() if p.grad is not None)
    return {**{k: v.item() for k, v in terms.items()}, "grads_finite": float(grads_finite)}

In [ ]:
check_d = {(name, domain): finite_terms(fresh, domain, config)
           for name, config in (("building", building_config), ("structure", structure_config)) for domain in DOMAINS}
assert all(math.isfinite(v) for terms in check_d.values() for v in terms.values()), "non-finite term"
assert all(terms["grads_finite"] == 1.0 for terms in check_d.values())
pd.DataFrame(check_d).round(4)

In [ ]:
check_d_reference = {(name, domain): {k: v.item() for k, v in structure_loss_terms(
                         *logits_of(reference[domain], domain), config).items()}
                     for name, config in (("building", building_config), ("structure", structure_config))
                     for domain in DOMAINS}
assert all(math.isfinite(v) for terms in check_d_reference.values() for v in terms.values())
pd.DataFrame(check_d_reference).round(4)